# Symbolic Fall Detector v4: test on URFD

The rules live in `fall_detection_models/code/stgcn_urfd/symbolic_v4.py`. Each fold is fitted on its own training videos only, then scored on its validation and test videos (`splits/split_scene_v2.json`). Open from `notebooks/` or the project root and **Run All**. Needs `processed_data/kinect_height/` (made by `kinect_neck_height.py`).

In [1]:
import os, sys, json
import numpy as np
PROJECT_ROOT = os.environ.get("PROJECT_ROOT") or ("." if os.path.isdir("splits") else "..")
CODE = os.path.join(PROJECT_ROOT, "fall_detection_models", "code", "stgcn_urfd")
sys.path.insert(0, CODE); os.environ["PROJECT_ROOT"] = os.path.abspath(PROJECT_ROOT)
import symbolic_v4 as S4
import fusion_v4_experiments as X
X.ROOT = os.path.abspath(PROJECT_ROOT)
X.NPY = os.path.join(X.ROOT, "processed_data", "cfg_d31bfed741ab", "urfd_npy")
X.KIN = os.path.join(X.ROOT, "processed_data", "kinect_height")
split, data = X.load_all()
print(len(data), "videos")

70 videos


In [2]:
models, out = {}, {}
for f in split["folds"]:
    k = f["fold"]
    m = S4.fit([(data[v]["h"], data[v]["y"], data[v]["l3"]) for v in f["train"]])
    models[k] = m
    c = m.config()
    print(f"fold {k}: tau_low={c['tau_low']:.3f} m, delta_fall={c['delta_fall']:.3f} m, delta_on={c['delta_on']:.3f} m")
    print("   P(fall | state):", {s: round(p, 3) for s, p in c["state_p"].items()})
    for part in ("val", "test"):
        for v in f[part]:
            out[(k, part, v)] = m.predict(data[v]["h"])

fold 0: tau_low=0.425 m, delta_fall=0.358 m, delta_on=0.179 m
   P(fall | state): {'UNKNOWN': 0.025, 'FALL': 0.998, 'DESCENDING': 0.389, 'UPRIGHT': 0.061, 'LOW_NOFALL': 0.036, 'MID': 0.056}


fold 1: tau_low=0.340 m, delta_fall=0.583 m, delta_on=0.291 m
   P(fall | state): {'UNKNOWN': 0.5, 'FALL': 0.995, 'DESCENDING': 0.398, 'UPRIGHT': 0.051, 'LOW_NOFALL': 0.5, 'MID': 0.074}


## Results
Frame-level metrics pooled over the two folds, events with the ST-GCN definitions (a fall counts if any frame from the onset on is positive; an ADL video is clean if no frame is positive).

In [3]:
def show(title, rows, probs=None):
    m = X.metrics(rows, probs)
    keys = ["MCC", "F1", "Precision", "Recall", "Specificity", "Accuracy", "MacroF1"] + [k for k in ("ROC_AUC", "PR_AUC", "ECE") if k in m]
    print(title, {k: round(m[k], 3) for k in keys},
          f"| falls {m['falls_detected']}/{m['falls']}, ADL clean {m['adl_clean']}/{m['adl']}, alarms/min {m['fa_per_min']:.2f}",
          f"| TP {m['tp']} FP {m['fp']} FN {m['fn']} TN {m['tn']}")

for part in ("val", "test"):
    rows = [(v, data[v]["y"], out[(f["fold"], part, v)]["hard"]) for f in split["folds"] for v in f[part]]
    probs = [out[(f["fold"], part, v)]["p"] for f in split["folds"] for v in f[part]]
    show(f"{part.upper():4s} pooled ", rows, probs)
    for f in split["folds"]:
        show(f"{part.upper():4s} fold {f['fold']}", [(v, data[v]["y"], out[(f["fold"], part, v)]["hard"]) for v in f[part]])

VAL  pooled  {'MCC': np.float64(0.316), 'F1': 0.387, 'Precision': 0.268, 'Recall': 0.695, 'Specificity': 0.76, 'Accuracy': 0.753, 'MacroF1': 0.616, 'ROC_AUC': 0.701, 'PR_AUC': 0.236, 'ECE': 0.176} | falls 6/6, ADL clean 0/11, alarms/min 14.83 | TP 298 FP 812 FN 131 TN 2575
VAL  fold 0 {'MCC': np.float64(0.316), 'F1': 0.307, 'Precision': 0.186, 'Recall': 0.882, 'Specificity': 0.698, 'Accuracy': 0.711, 'MacroF1': 0.562} | falls 2/2, ADL clean 0/6, alarms/min 14.15 | TP 149 FP 652 FN 20 TN 1504
VAL  fold 1 {'MCC': np.float64(0.415), 'F1': 0.524, 'Precision': 0.482, 'Recall': 0.573, 'Specificity': 0.87, 'Accuracy': 0.818, 'MacroF1': 0.706} | falls 4/4, ADL clean 0/5, alarms/min 16.06 | TP 149 FP 160 FN 111 TN 1071
TEST pooled  {'MCC': np.float64(0.498), 'F1': 0.577, 'Precision': 0.482, 'Recall': 0.718, 'Specificity': 0.863, 'Accuracy': 0.841, 'MacroF1': 0.739, 'ROC_AUC': 0.817, 'PR_AUC': 0.475, 'ECE': 0.065} | falls 30/30, ADL clean 0/40, alarms/min 20.33 | TP 1294 FP 1390 FN 509 TN 8743
T

In [4]:
# per room on the test folds
rooms = {}
for f in split["folds"]:
    for v in f["test"]:
        rooms.setdefault(data[v]["room"], []).append((v, data[v]["y"], out[(f["fold"], "test", v)]["hard"]))
for r, rows in sorted(rooms.items()):
    m = X.metrics(rows)
    print(f"{r}: videos {len(rows)}, frames {sum(len(x[1]) for x in rows)}, fall frames caught {m['tp']}/{m['tp']+m['fn']}, false-alarm frames {m['fp']}, ADL clean {m['adl_clean']}/{m['adl']}")

A4: videos 12, frames 3020, fall frames caught 0/0, false-alarm frames 289, ADL clean 0/12
A5: videos 11, frames 2180, fall frames caught 0/0, false-alarm frames 174, ADL clean 0/11
A6: videos 6, frames 585, fall frames caught 0/0, false-alarm frames 97, ADL clean 0/6
A7: videos 6, frames 2035, fall frames caught 0/0, false-alarm frames 455, ADL clean 0/6
A8: videos 5, frames 1121, fall frames caught 0/0, false-alarm frames 372, ADL clean 0/5
F1: videos 18, frames 1361, fall frames caught 673/882, false-alarm frames 3, ADL clean 0/0
F2: videos 12, frames 1634, fall frames caught 621/921, false-alarm frames 0, ADL clean 0/0


In [5]:
# save test-fold outputs (each video once, scored by the model of the fold whose training never saw its room)
save = {}
for f in split["folds"]:
    for v in f["test"]:
        o = out[(f["fold"], "test", v)]
        for key in ("state", "hard", "p", "support80", "upright"):
            save[f"{v}__{key}"] = o[key]
np.savez(os.path.join(PROJECT_ROOT, "symbolic_v4_predictions.npz"), **save)
print("saved symbolic_v4_predictions.npz")

saved symbolic_v4_predictions.npz
